# Fine-tune LFM2 Extract on lesson documents

Teaches `LiquidAI/LFM2-1.2B-Extract` to turn one section of a lesson document into lesson JSON, with a LoRA adapter, then merges it and converts it to ONNX for transformers.js.

Before you start, make the data on your machine:

```bash
cd packages/core
node scripts/extract-eval/make-dataset.mjs
```

and upload `train.jsonl` and `holdout.jsonl` from `scripts/extract-eval/out/dataset/` with the file panel on the left (or mount Drive and change the paths in the next cell).

Runtime: a free T4 works; training takes 30 to 60 minutes there (156 optimizer steps at 12 to 24 seconds each), 5 to 12 minutes on an A100 or L4, which can also use bfloat16; the config cell picks that up. The cells run on transformers 4.56 and later, including the 5.x that Colab installs today.

Fill in the three names in the first cell. Everything else runs top to bottom.

In [ ]:
import os

BASE_MODEL = "LiquidAI/LFM2-1.2B-Extract"
HF_USER = "YOUR_NAME"                         # your Hugging Face username
MERGED_REPO = f"{HF_USER}/LFM2-1.2B-Extract-lesson"       # merged weights
ONNX_REPO = f"{HF_USER}/LFM2-1.2B-Extract-lesson-ONNX"    # what run.mjs --models takes

TRAIN_FILE = "train.jsonl"
HOLDOUT_FILE = "holdout.jsonl"
OUTPUT_DIR = "lfm2-extract-lesson"

EPOCHS = 3
LEARNING_RATE = 2e-4
LORA_RANK = 16
MAX_LENGTH = 4096   # a section plus the schema is 1.5k to 2.5k tokens

# How many held-out sections the scoring cell generates. Each one is a full
# section of JSON, about a minute on a T4 without the fast kernel, so 20 is a
# 20-minute check; None scores all 84 (one to two hours on a T4).
EVAL_SECTIONS = 20

# Let the allocator grow segments instead of fragmenting; must be set before
# torch is imported. Worth a few hundred MB on a 15 GB T4.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

In [ ]:
# Colab ships an old torchao that transformers 5 refuses to import beside; this
# notebook never uses it, so it goes.
%pip uninstall -q -y torchao
%pip install -q "transformers>=4.56" "trl>=0.21" peft datasets accelerate huggingface_hub

In [ ]:
# The fast CUDA kernel for LFM2's convolution blocks. Without it transformers
# falls back to a PyTorch version that makes training about three times slower.
# The project publishes prebuilt wheels per torch and CUDA version; this asks
# for the exact one, then for the nearest older torch build, and checks that
# whatever it installed actually imports (in a separate process, so a wheel
# built against a different torch cannot break this session). It gives up
# rather than compiling for 20 minutes. It must run before transformers is
# imported.
import subprocess
import sys
import torch

RELEASE = "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0"
major, minor = (int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
cuda = "cu" + torch.version.cuda.split(".")[0]
abi = "TRUE" if torch.compiled_with_cxx11_abi() else "FALSE"
py = f"cp{sys.version_info.major}{sys.version_info.minor}"
pip = [sys.executable, "-m", "pip", "-q"]

def try_wheel(torch_mm):
    wheel = f"causal_conv1d-1.7.0+{cuda}torch{torch_mm}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
    if subprocess.run(pip + ["install", "--no-deps", f"{RELEASE}/{wheel}"]).returncode != 0:
        return False
    works = subprocess.run([sys.executable, "-c", "import causal_conv1d_cuda"], capture_output=True).returncode == 0
    if not works:
        subprocess.run(pip + ["uninstall", "-y", "causal-conv1d"])
    print(wheel, "imports" if works else "does not load against this torch")
    return works

installed = any(try_wheel(f"{major}.{m}") for m in range(minor, max(minor - 3, 5), -1))
print("fast kernel installed" if installed else "no usable prebuilt wheel; training will use the slower fallback")

## Load the model and the data

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "Pick a GPU runtime: Runtime > Change runtime type"
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(torch.cuda.get_device_name(0), "bfloat16" if BF16 else "float16")

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
try:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map="auto")
except TypeError:  # transformers 4.x spells it torch_dtype
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE, device_map="auto")

data = load_dataset("json", data_files={"train": TRAIN_FILE, "test": HOLDOUT_FILE})
data = data.select_columns(["messages"])
print(data)

The trainer needs the names of the linear layers to put adapters on. LFM2 mixes attention blocks with gated convolution blocks, so the names differ from a plain Llama; this cell finds them rather than guessing.

In [ ]:
import torch.nn as nn

names = sorted({n.split(".")[-1] for n, m in model.named_modules() if isinstance(m, nn.Linear)})
TARGET_MODULES = [n for n in names if n != "lm_head"]
print(TARGET_MODULES)

## Train

A checkpoint is written every epoch. If Colab disconnects, rerun the cells above and this one with `RESUME = True`.

In [ ]:
import math
import os
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

RESUME = False
GRAD_ACCUM = 8

# Warm up over the first 5% of steps, given as a step count: transformers 5.x
# dropped warmup_ratio, and an integer works on 4.x and 5.x alike.
WARMUP_STEPS = max(1, round(0.05 * EPOCHS * math.ceil(len(data["train"]) / GRAD_ACCUM)))

lora = LoraConfig(
    r=LORA_RANK,
    lora_alpha=2 * LORA_RANK,
    lora_dropout=0.05,
    target_modules=TARGET_MODULES,
    task_type="CAUSAL_LM",
)

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    # Evaluation at batch 1 too. The default of 8 pads eight sections to the
    # longest and runs out of memory on a T4 at the end of the first epoch,
    # before the checkpoint for that epoch is written.
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    max_length=MAX_LENGTH,
    gradient_checkpointing=True,
    bf16=BF16,
    fp16=not BF16,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=data["train"],
    eval_dataset=data["test"],
    processing_class=tok,
    peft_config=lora,
)

resume = RESUME and os.path.isdir(OUTPUT_DIR) and any(d.startswith("checkpoint-") for d in os.listdir(OUTPUT_DIR))
trainer.train(resume_from_checkpoint=resume or None)

## Look at one held-out section

The three things the stock model got wrong: paragraphs split into sentences, keys of its own instead of the schema's, and answers invented for open questions. All three should be gone.

In [ ]:
import json

def extract(messages, max_new_tokens=1500):
    # Render the template to text first: transformers 5.x returns a dict from
    # apply_chat_template where 4.x returned a tensor, and this works on both.
    text = tok.apply_chat_template(messages[:2], add_generation_prompt=True, tokenize=False)
    prompt = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][prompt.shape[1]:], skip_special_tokens=True)

model.eval()
row = data["test"][0]
reply = extract(row["messages"])
print(reply[:2500])
print("\n--- expected ---")
expected = json.loads(row["messages"][2]["content"])
print(json.dumps({k: expected[k] for k in ("paragraphs", "spellingWords")}, indent=1)[:1200])
print(json.dumps(expected["questions"][:3], indent=1))

## Score held-out sections

A rough version of what `score.mjs` does, enough to decide whether to export: did the JSON parse, did every spelling word come back, how many prompts were found, how many answers match exactly, and were any answers invented for questions that print none. It scores the first `EVAL_SECTIONS` held-out sections (set in the first cell), spread across both held-out lessons, and shows progress as it goes.

In [ ]:
import re
import time

def norm(s):
    s = re.sub(r"(\d),(\d)", r"\1\2", str(s).lower())
    return " ".join(re.sub(r"[^\w\s]", " ", s).split())

def parse(reply):
    reply = reply.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    start = reply.find("{")
    try:
        return json.loads(reply[start:])
    except Exception:
        return None

# Every EVAL_SECTIONS-th row, so a sample still covers both held-out lessons and
# every layout rather than the first lesson in one layout.
total = len(data["test"])
count = total if EVAL_SECTIONS is None else min(EVAL_SECTIONS, total)
step = max(1, total // count)
rows = data["test"].select(range(0, step * count, step))

totals = dict(sections=0, parsed=0, spelling_ok=0, prompts=0, prompts_found=0, answers=0, answers_ok=0, invented=0)
started = time.time()
for i, row in enumerate(rows, 1):
    expected = json.loads(row["messages"][2]["content"])
    got = parse(extract(row["messages"]))
    totals["sections"] += 1
    if got:
        totals["parsed"] += 1
        if {norm(w) for w in got.get("spellingWords", [])} == {norm(w) for w in expected["spellingWords"]}:
            totals["spelling_ok"] += 1
        by_prompt = {norm(q.get("prompt", "")): q for q in got.get("questions", []) if isinstance(q, dict)}
        for q in expected["questions"]:
            totals["prompts"] += 1
            match = by_prompt.get(norm(q["prompt"]))
            if not match:
                continue
            totals["prompts_found"] += 1
            got_answers = {norm(a) for a in (match.get("answers") or [])}
            want = {norm(a) for a in q["answers"]}
            if want:
                totals["answers"] += 1
                totals["answers_ok"] += got_answers == want
            elif got_answers:
                totals["invented"] += 1
    elapsed = time.time() - started
    print(f"{i}/{count} sections, {elapsed / i:.0f}s each, about {elapsed / i * (count - i) / 60:.0f} min left", end="\r")

pct = lambda a, b: f"{100 * a / b:.0f}%" if b else "n/a"
print()
print("sections parsed  ", pct(totals["parsed"], totals["sections"]))
print("spelling exact   ", pct(totals["spelling_ok"], totals["sections"]))
print("prompts found    ", pct(totals["prompts_found"], totals["prompts"]))
print("answers exact    ", pct(totals["answers_ok"], totals["answers"]))
print("invented answers ", totals["invented"], "(should be 0)")

The bar before this goes near the import: prompts and spelling above 95%, answers above 90%, invented answers at 0. If answers are still weak, add layouts to `layouts.mjs` and regenerate the data rather than adding epochs; 412 examples is thin for a 1.2B model.

## Merge the adapter and push the merged model

In [ ]:
from huggingface_hub import login

login()

merged = trainer.model.merge_and_unload()
merged.save_pretrained("merged", safe_serialization=True)
tok.save_pretrained("merged")
merged.push_to_hub(MERGED_REPO, private=True)
tok.push_to_hub(MERGED_REPO, private=True)
print("pushed", MERGED_REPO)

## Convert to ONNX for transformers.js

The onnx-community LFM2 files that transformers.js loads are made with Microsoft's onnxruntime-genai model builder, which supports LFM2 and takes a local checkpoint. Its raw output needs three small fixes before transformers.js will load it (cache names, a symbolic cache dimension, the chat template's location), which `relayout-onnx.py` applies. Two builds: `-e cpu` gives the q4 file `run.mjs` uses, `-e webgpu` the q4f16 file the app would ship.

Neither optimum-onnx (no LFM2 support) nor Liquid's own LiquidONNX (its output targets onnxruntime-genai, not transformers.js) does this on its own.

In [ ]:
%pip install -q onnxruntime-genai onnx onnx_ir onnxscript

import subprocess, sys

# cpu -> q4 (fp32 activations), what run.mjs runs through onnxruntime-node and what
#        the browser's wasm backend runs. That backend has no kernel for the int8
#        embedding lookup the builder emits for a tied embedding, so the embedding
#        is left as a plain table here (shared_embeddings=false), as in the
#        onnx-community files. Verified in Chromium.
# webgpu -> q4f16 (fp16 activations), what the app would ship. The WebGPU backend
#        runs the quantised embedding fine, so the smaller default build is kept.
for provider, extra in (("cpu", ["--extra_options", "shared_embeddings=false"]), ("webgpu", [])):
    print("building int4 for", provider)
    subprocess.run(
        [sys.executable, "-m", "onnxruntime_genai.models.builder",
         "-i", "merged", "-o", f"/content/build-{provider}", "-p", "int4", "-e", provider, "-c", "/content/hf-cache", *extra],
        check=True,
    )

In [ ]:
import urllib.request

# The relayout script from the repo (packages/core/scripts/extract-eval/relayout-onnx.py).
# Upload it beside the notebook with the file panel; failing that, this fetches it
# from main, which works once the branch carrying it has been merged.
RELAYOUT = "relayout-onnx.py"
if not os.path.exists(RELAYOUT):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Spelling-Creator/spelling-creator/main/packages/core/scripts/extract-eval/relayout-onnx.py",
        RELAYOUT,
    )

DEST = "/content/onnx-repo"
for provider, dtype in (("cpu", "q4"), ("webgpu", "q4f16")):
    subprocess.run([sys.executable, RELAYOUT, f"/content/build-{provider}", DEST, dtype, "merged"], check=True)
print(os.listdir(DEST), os.listdir(f"{DEST}/onnx"))

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(ONNX_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=DEST, repo_id=ONNX_REPO)
print("pushed", ONNX_REPO)

## Measure it with the real scorer

Back on your machine, against the two lessons that were held out of training:

```bash
cd packages/core
node scripts/extract-eval/run.mjs --lessons 2 --sections 6 --styles all --models YOUR_NAME/LFM2-1.2B-Extract-lesson-ONNX
```

A private repo needs `HF_TOKEN` in the environment for transformers.js to fetch it. Compare the rows with the stock ones in the docs page (Monorepo, Document import experiment).